In [ ]:
%pip install langchain-groq groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.1/591.1 kB 6.6 MB/s  0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.20
    Uninstalling langchain-protocol-0.0.20:
      Successfully uninstalled langchain-protocol-0.0.20
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-groq]


In [ ]:
from langchain_groq import ChatGroq

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [ ]:
SYSTEM = """
Role:
You are CareerPilot AI, an intelligent career
and placement assistant for engineering students.

Facts:
Students have different skills and career goals.
Different careers require different technical skills.
Learning plans should be personalized.

Rules:
Never guarantee a job or salary.
Never invent job openings.
Ask for missing information when necessary.
If you do not know something, say so.

Format:
Reply professionally in simple English
using under 60 words.
"""

In [ ]:
CAREERS = {
    "Machine Learning Engineer": [
        "Python", "SQL", "Statistics",
        "Machine Learning", "Deep Learning",
        "MLOps", "Git"
    ],

    "Data Scientist": [
        "Python", "SQL", "Statistics",
        "Machine Learning", "Data Visualization"
    ],

    "AI Engineer": [
        "Python", "LLMs", "LangChain",
        "RAG", "API Integration", "Tool Calling"
    ]
}

print(CAREERS)

{'Machine Learning Engineer': ['Python', 'SQL', 'Statistics', 'Machine Learning', 'Deep Learning', 'MLOps', 'Git'], 'Data Scientist': ['Python', 'SQL', 'Statistics', 'Machine Learning', 'Data Visualization'], 'AI Engineer': ['Python', 'LLMs', 'LangChain', 'RAG', 'API Integration', 'Tool Calling']}


In [ ]:
messages = [
    ("system", SYSTEM),

    ("human", "I know Python and SQL."),
    ("ai", "Which career are you interested in?"),

    ("human", "Machine Learning Engineer."),
    ("ai", "You should study statistics and machine learning."),

    ("human", "I know Python and want to become a Data Scientist.")
]

response = llm.invoke(messages)

print(response.content)

Great! To tailor a plan, could you tell me:

1. Your current level with statistics, linear algebra, and SQL (beginner, intermediate, advanced).  
2. Any experience with data cleaning/visualization tools (e.g., pandas, matplotlib, Tableau).  
3. How many hours per week you can study and your preferred format (online courses, books, projects).


In [ ]:
import json

prompt = """
Extract student information from the following text.

Return only valid JSON with these fields:
name, year, skills, career_goal.

Student information:
My name is Reddy. I am a second-year student.
I know Python and SQL.
I want to become a Machine Learning Engineer.
"""

response = llm.invoke(prompt)

print(response.content)

{
  "name": "Reddy",
  "year": "second-year",
  "skills": ["Python", "SQL"],
  "career_goal": "Machine Learning Engineer"
}


In [ ]:
history = [("system", SYSTEM)]

def chat(question):
    history.append(("human", question))

    response = llm.invoke(history).content

    history.append(("ai", response))

    return response

In [ ]:
print(chat("My name is Reddy. I know Python and SQL."))

Nice to meet you, Reddy!  
To suggest suitable career paths and a learning plan, could you tell me:

1. Your engineering discipline and graduation year  
2. Areas of interest (e.g., data analysis, web development, AI, finance)  
3. Any preferred industry or role type  

Thanks!


In [ ]:
print(chat("What skills do I already know?"))

Based on what you’ve shared, your current technical skills are:

- **Python** – programming and scripting  
- **SQL** – database querying and management  

If you have additional skills (e.g., data visualization, cloud platforms, etc.), let me know so I can tailor recommendations.


In [ ]:
def analyze_skills(career: str, skills: list[str]) -> dict:

    if career not in CAREERS:
        return {"error": "Career not found"}

    required = CAREERS[career]

    known = {skill.lower().strip() for skill in skills}

    matched = [
        skill for skill in required
        if skill.lower() in known
    ]

    missing = [
        skill for skill in required
        if skill.lower() not in known
    ]

    coverage = len(matched) / len(required) * 100

    return {
        "career": career,
        "matched_skills": matched,
        "missing_skills": missing,
        "skill_coverage": round(coverage, 2)
    }

In [ ]:
result = analyze_skills(
    "Machine Learning Engineer",
    ["Python", "SQL", "Git"]
)

print(json.dumps(result, indent=4))

{
    "career": "Machine Learning Engineer",
    "matched_skills": [
        "Python",
        "SQL",
        "Git"
    ],
    "missing_skills": [
        "Statistics",
        "Machine Learning",
        "Deep Learning",
        "MLOps"
    ],
    "skill_coverage": 42.86
}


In [ ]:
from functools import wraps

def log_tool(func):

    @wraps(func)
    def wrapper(*args, **kwargs):
        print("Executing:", func.__name__)
        return func(*args, **kwargs)

    return wrapper


@log_tool
def get_careers():
    return list(CAREERS.keys())


print(get_careers())

Executing: get_careers
['Machine Learning Engineer', 'Data Scientist', 'AI Engineer']


In [ ]:
def safe_ask(question):
    try:
        return chat(question)

    except Exception as e:
        print("Error type:", type(e).__name__)
        return "Something went wrong. Please try again."


print(safe_ask("How can I improve my Python skills?"))

1. **Build projects** – create a web scraper, a Flask/Django app, or a data‑analysis notebook.  
2. **Practice algorithms** – use LeetCode, HackerRank, or CodeSignal daily.  
3. **Read code** – explore open‑source repos on GitHub related to your interests.  
4. **Take a course** – advanced topics like OOP, testing, or async (Coursera, Udemy, freeCodeCamp).  
5. **Join a community** – attend local meetups or Discord groups to get feedback and mentorship.
